### RAG pipeline - Data ingestion to Vector Db Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/shivamtripathi/Documents/agentic ai/KrishNayak/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### read all the PDF  inside directory 

def process_all_pdfs(pdf_directory):
    """ Process all pdf files inside directory """
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF to process")
    for pdf_file in pdf_files:
        print(f"\nprocessing : {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']="pdf"
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"error is {e}")
    print(f"Total documents processed are {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")

found 2 PDF to process

processing : github-foundations-exam-study-guide.pdf
loaded 8 pages

processing : shivam_tripathi_v2.pdf
loaded 1 pages
Total documents processed are 9


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 14.1 (Build 23B74) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20231101201134Z00'00'", 'title': 'github-foundations-exam-study-guide', 'moddate': "D:20231101201134Z00'00'", 'source': '../data/pdf/github-foundations-exam-study-guide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'github-foundations-exam-study-guide.pdf', 'file_type': 'pdf'}, page_content='Study Guide \nGitHub Foundations\nGet exam-ready for your GitHub Foundations Certification \nwith our comprehensive study guide. We’ve curated the \nessential resources and insights you need to navigate the \nfoundations of GitHub and boost your success with the exam.\nObjective Domains \nAn objective domain for a certification exam, often referred to as a “domain” or “exam domain,” is a \nstructured outline or framework that defines the specific knowledge, skills, and topics that the certification \nexam will cover . It provides a clear roadmap for wha

In [4]:
### TExt Splitting into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ split the documents into chunks for better RAG Performance """
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("example chunk ")
        print(f"content :{split_docs[0].page_content[:200]}...")
        print(f"metadata : {split_docs[0].metadata}")
    return split_docs   


In [5]:
chunks=split_documents(all_pdf_documents)

split 9 documents into 20 chunks
example chunk 
content :Study Guide 
GitHub Foundations
Get exam-ready for your GitHub Foundations Certification 
with our comprehensive study guide. We’ve curated the 
essential resources and insights you need to navigate t...
metadata : {'producer': 'macOS Version 14.1 (Build 23B74) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20231101201134Z00'00'", 'title': 'github-foundations-exam-study-guide', 'moddate': "D:20231101201134Z00'00'", 'source': '../data/pdf/github-foundations-exam-study-guide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'github-foundations-exam-study-guide.pdf', 'file_type': 'pdf'}


### Embedding and Vectore store DB 


In [6]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List , Dict,Any, Tuple 
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """ Load sentence transformer model """
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully , Model dimensions are {self.model.get_embedding_dimension()}")
        except Exception as e :
            print(f"exception occured :{e}")
            raise e
    def generate_embeddings(self,texts:List[str])-> np.ndarray:
        """
        generate Embeddings for list of text 

        ARGS:
            texts: List of text strings to embed 
        return :
        numpy array of embedding of shape (len(texts),embedding_dimensions)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate embeddings for {len(texts)}...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"GEnerated embedding with shape {embeddings.shape}")
        return embeddings
    
emnbedding_manager=EmbeddingManager()
emnbedding_manager



Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8046.14it/s]


Model loaded successfully , Model dimensions are 384


### Vector Store


In [8]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        initialise vector store 
        
        Args:
            collection name =name of the chromadb collection 
            persist_directory = Directory to persist vector store  
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()
        
    def _initialise_store(self):
        """initialise chroma db client and collection"""
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection= self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for rag"}
            )
            print(f"Vector store initialised , collection f{self.collection_name}")
            print(f"existing documents in collection {self.collection.count()}")

        except Exception as e:
            print(f"error occureed  while initialising vector store,{e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        add documents and there embeedding to vector store 
        ARGs:
            documents: list of langchain documents
            embedding : corresponding embedding for the documents

        """
        if len(documents)!= len(embeddings):
            raise ValueError("number of documents must match number of embeddings")
        print(f"adding {len(documents)} documents to the vector store ")
        
        # prepare for chromaDB 
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # generate unique id 
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare  metadata 
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #documents content

            documents_text.append(doc.page_content)

            # Embeddings 
            embeddings_list.append(embedding.tolist())
        
        # add to collection 
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
                )
            print(f"successfully added {len(documents)} documetns to vector store ")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print("error inserting document sint vector store ")
            raise
vectorstore=VectorStore()
vectorstore




Vector store initialised , collection fpdf_documents
existing documents in collection 20


In [9]:
chunks

[Document(metadata={'producer': 'macOS Version 14.1 (Build 23B74) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20231101201134Z00'00'", 'title': 'github-foundations-exam-study-guide', 'moddate': "D:20231101201134Z00'00'", 'source': '../data/pdf/github-foundations-exam-study-guide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'github-foundations-exam-study-guide.pdf', 'file_type': 'pdf'}, page_content='Study Guide \nGitHub Foundations\nGet exam-ready for your GitHub Foundations Certification \nwith our comprehensive study guide. We’ve curated the \nessential resources and insights you need to navigate the \nfoundations of GitHub and boost your success with the exam.\nObjective Domains \nAn objective domain for a certification exam, often referred to as a “domain” or “exam domain,” is a \nstructured outline or framework that defines the specific knowledge, skills, and topics that the certification \nexam will cover . It provides a clear roadmap for wha

In [11]:
### convert the text to embeddings 

texts=[doc.page_content for doc in chunks]
### generate the embeddings 
embeddings=emnbedding_manager.generate_embeddings(texts)

### store in the vector database 
vectorstore.add_documents(chunks,embeddings)



Generate embeddings for 20...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

GEnerated embedding with shape (20, 384)
adding 20 documents to the vector store 
successfully added 20 documetns to vector store 
Total documents in collection: 60


### Retrival pipeline from vector store

In [ ]:
class RagRetriver:
    """
    handles query based retrival from vector store 
    """
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager): 
        """
        Initialise  the retriver 

        args :
            vector_store: vector store containing the document embeddings
            embedding manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str:Any]]:
        """
        Retrieve relevent documents for the query 
        args :

            query : the search query 
            top_k: NUmber of top results to show 
            score_threshold: Minimum similarity score threshold 
        
        returns :
            List of dictionaries containing retrived documents and metadata 
        
        """
        print(f"retrieving  document for the query {query}")
        print(f"top_k={top_k}, and score_threshold is = {score_threshold}")
        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas= results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i , (doc_id, document,metadata, distance) in enumerate(zip(ids,documents, metadatas,distances)):
                    similarity_score=1-distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"we retrieved {len(retrieved_docs)} documents after filtering ")

            else:
                print(" no documents found ")
            return retrieved_docs
        except Exception as e :
            print(f"found error as {e}")
            return []

    


SyntaxError: incomplete input (3266940392.py, line 63)